# Tres en raya con MDP (3_raya_markov)

Esta notebook crea un agente para tres en raya usando un enfoque de proceso de decision de Markov (MDP).
El agente aprende una funcion de valor, la guarda en `3_raya_markov.pickle` y luego la usa para jugar contra una persona.

Idea general:
- Estado: tablero 3x4.
- Accion: elegir una casilla libre.
- Transicion: despues de mover el agente, responde un oponente aleatorio.
- Recompensa: +1 si gana el agente, -1 si pierde, 0 en empate o estados intermedios.
- Politica: elegir la accion con mayor valor esperado segun Bellman.

In [ ]:
import numpy as np
import pickle
import random

## Entorno

El tablero es de 3 filas por 4 columnas y la victoria ocurre cuando un jugador logra 3 en raya en horizontal, vertical o diagonal.

In [ ]:
class Board:
    def __init__(self, rows=3, cols=4, win_len=3):
        self.rows = rows
        self.cols = cols
        self.win_len = win_len
        self.state = np.zeros((rows, cols), dtype=int)

    def clone(self):
        board = Board(self.rows, self.cols, self.win_len)
        board.state = self.state.copy()
        return board

    def reset(self):
        self.state = np.zeros((self.rows, self.cols), dtype=int)

    def valid_moves(self):
        return [(r, c) for r in range(self.rows) for c in range(self.cols) if self.state[r, c] == 0]

    def update(self, symbol, row, col):
        if self.state[row, col] != 0:
            raise ValueError('movimiento ilegal')
        self.state[row, col] = symbol

    def winner(self):
        directions = [(0, 1), (1, 0), (1, 1), (-1, 1)]
        for r in range(self.rows):
            for c in range(self.cols):
                symbol = self.state[r, c]
                if symbol == 0:
                    continue
                for dr, dc in directions:
                    end_r = r + dr * (self.win_len - 1)
                    end_c = c + dc * (self.win_len - 1)
                    if not (0 <= end_r < self.rows and 0 <= end_c < self.cols):
                        continue
                    total = 0
                    for k in range(self.win_len):
                        total += self.state[r + dr * k, c + dc * k]
                    if total == symbol * self.win_len:
                        return int(symbol)
        if not self.valid_moves():
            return 0
        return None

    def state_key(self):
        return tuple(int(x) for x in self.state.reshape(-1))

## Agente MDP

El agente aprende una funcion de valor `V(s)` con respaldos de Bellman.
Para cada accion, se calcula el valor esperado suponiendo que el oponente responde de forma aleatoria.

In [ ]:
class MDPAgent:
    def __init__(self, alpha=0.3, gamma=0.95, epsilon=0.4, epsilon_min=0.05, epsilon_decay=0.9995, symbol=1):
        self.values = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.symbol = symbol
        self.opponent_symbol = -symbol

    def get_value(self, board):
        return self.values.get(board.state_key(), 0.0)

    def reward_from_result(self, result):
        if result == self.symbol:
            return 1.0
        if result == self.opponent_symbol:
            return -1.0
        return 0.0

    def transition(self, board, action):
        next_board = board.clone()
        next_board.update(self.symbol, action[0], action[1])
        result = next_board.winner()
        if result is not None:
            return next_board, self.reward_from_result(result), True

        opponent_moves = next_board.valid_moves()
        if not opponent_moves:
            return next_board, 0.0, True

        opp_action = random.choice(opponent_moves)
        next_board.update(self.opponent_symbol, opp_action[0], opp_action[1])
        result = next_board.winner()
        if result is not None:
            return next_board, self.reward_from_result(result), True

        return next_board, 0.0, False

    def action_value(self, board, action):
        next_board = board.clone()
        next_board.update(self.symbol, action[0], action[1])
        result = next_board.winner()
        if result is not None:
            return self.reward_from_result(result)

        opponent_moves = next_board.valid_moves()
        if not opponent_moves:
            return 0.0

        total = 0.0
        for opp_action in opponent_moves:
            after_opp = next_board.clone()
            after_opp.update(self.opponent_symbol, opp_action[0], opp_action[1])
            result = after_opp.winner()
            if result is not None:
                total += self.reward_from_result(result)
            else:
                total += self.get_value(after_opp)
        return total / len(opponent_moves)

    def best_action(self, board):
        valid_moves = board.valid_moves()
        if not valid_moves:
            raise ValueError('no hay movimientos validos')

        best_action = valid_moves[0]
        best_value = -1e9
        for action in valid_moves:
            value = self.action_value(board, action)
            if value >= best_value:
                best_value = value
                best_action = action
        return best_action, best_value

    def choose_action(self, board):
        valid_moves = board.valid_moves()
        if random.random() < self.epsilon:
            return random.choice(valid_moves)
        return self.best_action(board)[0]

    def update_state_value(self, board, target):
        key = board.state_key()
        old_value = self.values.get(key, 0.0)
        self.values[key] = old_value + self.alpha * (target - old_value)

    def train(self, episodes=2000, log_every=250):
        history = []
        for episode in range(1, episodes + 1):
            board = Board()
            done = False
            steps = 0
            while not done and steps < 20:
                action = self.choose_action(board)
                next_board, reward, done = self.transition(board, action)
                target = reward if done else reward + self.gamma * self.get_value(next_board)
                self.update_state_value(board, target)
                board = next_board
                steps += 1

            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

            if log_every and episode % log_every == 0:
                info = (episode, len(self.values), round(self.epsilon, 4))
                history.append(info)
                print(f'episode={episode} estados={len(self.values)} epsilon={self.epsilon:.4f}')

        return history

## Entrenamiento y guardado

Aqui se entrena el agente y se guarda la funcion de valor en un archivo pickle.

In [ ]:
random.seed(7)
np.random.seed(7)

agent = MDPAgent(alpha=0.35, gamma=0.95, epsilon=0.5, epsilon_min=0.05, epsilon_decay=0.9995, symbol=1)
history = agent.train(episodes=2000, log_every=250)

items = sorted(agent.values.items(), key=lambda kv: kv[1], reverse=True)
print('estados en la tabla:', len(items))
for state, value in items[:10]:
    print(value, state)

payload = {
    'values': agent.values,
    'rows': 3,
    'cols': 4,
    'win_len': 3,
    'symbol': agent.symbol,
    'gamma': agent.gamma
}

with open('3_raya_markov.pickle', 'wb') as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

print('Guardado 3_raya_markov.pickle')

## Jugar contra el agente

Esta parte carga el pickle y permite jugar contra el agente entrenado.

In [ ]:
class LoadedMDPAgent(MDPAgent):
    def __init__(self, payload):
        super().__init__(alpha=0.0, gamma=payload.get('gamma', 0.95), epsilon=0.0, symbol=payload.get('symbol', 1))
        self.values = payload['values']
        self.rows = payload.get('rows', 3)
        self.cols = payload.get('cols', 4)
        self.win_len = payload.get('win_len', 3)

def draw_board(board):
    symbols = {1: 'X', -1: 'O', 0: ' '}
    print('')
    for r in range(board.rows):
        row_values = []
        for c in range(board.cols):
            value = board.state[r, c]
            if value == 0:
                row_values.append(str(r * board.cols + c + 1))
            else:
                row_values.append(symbols[int(value)])
        print(' ' + ' | '.join(row_values))
        if r < board.rows - 1:
            print('---+' * (board.cols - 1) + '---')
    print('')

def human_move(board):
    valid_numbers = {r * board.cols + c + 1: (r, c) for r, c in board.valid_moves()}
    while True:
        entry = input(f'Tu turno (elige {sorted(valid_numbers.keys())}): ').strip()
        if not entry.isdigit():
            print('Entrada invalida. Escribe un numero.')
            continue
        number = int(entry)
        if number not in valid_numbers:
            print('Esa casilla no esta disponible.')
            continue
        return valid_numbers[number]

def play_game(pickle_file='3_raya_markov.pickle', human_starts=False):
    with open(pickle_file, 'rb') as f:
        payload = pickle.load(f)

    agent = LoadedMDPAgent(payload)
    board = Board(rows=payload.get('rows', 3), cols=payload.get('cols', 4), win_len=payload.get('win_len', 3))
    human_symbol = -1

    print('Humano: O | Agente: X')
    draw_board(board)

    agent_turn = not human_starts
    while board.winner() is None:
        if agent_turn:
            row, col = agent.best_action(board)[0]
            board.update(agent.symbol, row, col)
            print(f'Agente juega en casilla {row * board.cols + col + 1}')
        else:
            row, col = human_move(board)
            board.update(human_symbol, row, col)

        draw_board(board)
        agent_turn = not agent_turn

    result = board.winner()
    if result == agent.symbol:
        print('Gana el agente.')
    elif result == human_symbol:
        print('Ganaste.')
    else:
        print('Empate.')

## Notas finales

- El agente aprende una funcion de valor y elige la jugada que maximiza el valor esperado.
- El oponente se modela como aleatorio, lo que hace que el problema sea tratable como MDP.
- Si quieres mas calidad, aumenta `episodes` en el entrenamiento.